# M01-P01 — Exploring Bedrock LLMs

Trabajaremos directamente con **Amazon Bedrock Converse API**, sin LangChain. El objetivo es observar cómo cambian las respuestas al modificar `system prompt`, temperatura, `maxTokens`, contexto y streaming.

> Si la sesión SSO ha caducado: `aws sso login --profile genai-course --use-device-code`


## 1. Cliente e identidad


In [ ]:
import json, time
import boto3
import pandas as pd
from enterprise_genai.config import get_settings

settings = get_settings()
session = boto3.Session(profile_name=settings.aws_profile or None, region_name=settings.aws_region)
identity = session.client("sts").get_caller_identity()
runtime = session.client("bedrock-runtime", region_name=settings.aws_region)
MODEL_ID = settings.bedrock_text_model_id

print("AWS identity:", identity["Arn"])
print("Region:", settings.aws_region)
print("Model:", MODEL_ID)


## 2. Primera llamada — abrir la caja


In [ ]:
incident = (
    "Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. "
    "El servicio web corporativo funciona correctamente. "
    "El equipo de red informa de errores intermitentes en el backend de autenticación."
)

request = {
    "modelId": MODEL_ID,
    "system": [{"text": "Eres un asistente de soporte empresarial. Resume incidencias de forma clara y no inventes datos."}],
    "messages": [{"role": "user", "content": [{"text": f"Resume esta incidencia en 3 viñetas:\n\n{incident}"}]}],
    "inferenceConfig": {"temperature": 0.2, "maxTokens": 200},
}
print(json.dumps(request, indent=2, ensure_ascii=False))

response = runtime.converse(**request)
print("\n" + response["output"]["message"]["content"][0]["text"])
print("\nStop reason:", response.get("stopReason"))
print("Usage:", response.get("usage"))
print("Metrics:", response.get("metrics"))


### Observa

`inputTokens`, `outputTokens`, `totalTokens`, `latencyMs` y `stopReason`. Estas métricas permiten hablar con datos reales de coste, rendimiento y observabilidad.


## 3. Banco de pruebas


In [ ]:
def run_experiment(label, user_message, *, system_prompt=None, temperature=0.2, max_tokens=200, model_id=MODEL_ID):
    kwargs = {
        "modelId": model_id,
        "messages": [{"role": "user", "content": [{"text": user_message}]}],
        "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
    }
    if system_prompt:
        kwargs["system"] = [{"text": system_prompt}]

    started = time.perf_counter()
    response = runtime.converse(**kwargs)
    elapsed_ms = round((time.perf_counter() - started) * 1000)
    usage = response.get("usage", {})
    metrics = response.get("metrics", {})

    return {
        "label": label,
        "text": response["output"]["message"]["content"][0]["text"],
        "input_tokens": usage.get("inputTokens"),
        "output_tokens": usage.get("outputTokens"),
        "bedrock_latency_ms": metrics.get("latencyMs"),
        "client_elapsed_ms": elapsed_ms,
        "stop_reason": response.get("stopReason"),
    }


## 4. Experimento A — `system prompt`

Aislamos **una sola variable**: el `system prompt`. Las respuestas se piden breves para evitar que `maxTokens` contamine el experimento. Lo normal es que ambas terminen en `end_turn`.


In [ ]:
user_message = (
    "Analiza esta incidencia en 3 puntos breves. "
    "Indica qué información consideras más relevante y qué harías a continuación.\n\n" + incident
)

variants = [
    ("Sin system prompt", None),
    ("Coordinador NOC",
     "Actúa como coordinador de un NOC empresarial. "
     "Responde exactamente con tres apartados: Impacto, Hipótesis y Siguiente acción. "
     "Una frase breve por apartado. No afirmes una causa raíz si no está demostrada.")
]

results = []
for label, system_prompt in variants:
    r = run_experiment(label, user_message, system_prompt=system_prompt, temperature=0.2, max_tokens=180)
    results.append(r)
    print(f"\n--- {label} ---")
    print(r["text"])
    print("Stop reason:", r["stop_reason"])

pd.DataFrame(results)[["label","input_tokens","output_tokens","bedrock_latency_ms","stop_reason"]]


**Modifica** el segundo `system prompt`: ejecutivo, técnico N1 o responsable de seguridad. Los datos no cambian; debe cambiar el comportamiento.

Si alguna respuesta termina en `max_tokens`, aumenta temporalmente el límite: ese no es el fenómeno que queremos estudiar aquí.


## 5. Experimento B — temperatura

Valores bajos suelen producir salidas más previsibles; valores altos permiten más variación. Para verlo mejor repetimos la misma tarea.


In [ ]:
temp_results = []
for temperature in [0.0, 0.8]:
    for run in range(1, 4):
        r = run_experiment(
            f"T={temperature} run={run}",
            "Propón 5 títulos breves y profesionales para esta incidencia:\n\n" + incident,
            system_prompt="Eres un redactor técnico de un centro de operaciones.",
            temperature=temperature,
            max_tokens=180,
        )
        temp_results.append(r)
        print(f"\n--- T={temperature} / run={run} ---")
        print(r["text"])

pd.DataFrame(temp_results)[["label","output_tokens","bedrock_latency_ms","stop_reason"]]


Prueba también `temperature=0.4`. No existe una temperatura universalmente correcta: depende del caso de uso.


## 6. Experimento C — `maxTokens`

Ahora sí queremos provocar truncamiento. Mantenemos el resto y cambiamos solo el límite.


In [ ]:
length_results = []
for max_tokens in [40, 300]:
    r = run_experiment(
        f"maxTokens={max_tokens}",
        "Redacta un informe con: Resumen, Impacto, Evidencias, Hipótesis y Próximos pasos.\n\n" + incident,
        system_prompt="No inventes información que no aparezca en la incidencia.",
        temperature=0.2,
        max_tokens=max_tokens,
    )
    length_results.append(r)
    print(f"\n--- maxTokens={max_tokens} ---")
    print(r["text"])
    print("Stop reason:", r["stop_reason"])

pd.DataFrame(length_results)[["label","output_tokens","bedrock_latency_ms","stop_reason"]]


Esperamos que el límite pequeño termine normalmente en `max_tokens` y el amplio en `end_turn`. `maxTokens` es un **límite duro**, no una instrucción de estilo.


## 7. Experimento D — contexto

Todavía no hacemos RAG. Añadimos manualmente contexto corporativo para observar la diferencia entre **modelo solo** y **modelo + información empresarial**.


In [ ]:
question = "Para un P1 de VPN de un cliente premium, ¿cuál es el tiempo objetivo de respuesta y qué equipo coordina?"

without_context = run_experiment(
    "Sin contexto", question,
    system_prompt="Si faltan datos corporativos, dilo claramente.",
    temperature=0.0, max_tokens=140
)

context = (
    "Procedimiento OPS-INC-001:\n"
    "- Cliente premium + P1: respuesta inicial en 30 minutos.\n"
    "- La coordinación inicial corresponde al NOC.\n"
    "- Seguridad participa si hay indicadores de compromiso."
)

with_context = run_experiment(
    "Con contexto",
    f"Usa exclusivamente este contexto:\n\n{context}\n\nPregunta: {question}",
    system_prompt="No inventes políticas que no aparezcan en el contexto.",
    temperature=0.0, max_tokens=140
)

for r in [without_context, with_context]:
    print(f"\n--- {r['label']} ---")
    print(r["text"])


La idea que reutilizaremos en RAG es:

`pregunta + contexto relevante → respuesta fundamentada`


## 8. Experimento E — streaming


In [ ]:
stream_request = {
    "modelId": MODEL_ID,
    "messages": [{"role":"user","content":[{"text":"Explica brevemente por qué un fallo de autenticación VPN puede afectar a muchos usuarios."}]}],
    "inferenceConfig": {"temperature":0.2,"maxTokens":180},
}

started = time.perf_counter()
first_chunk_ms = None
response = runtime.converse_stream(**stream_request)

for event in response["stream"]:
    if "contentBlockDelta" in event and "text" in event["contentBlockDelta"]["delta"]:
        if first_chunk_ms is None:
            first_chunk_ms = round((time.perf_counter() - started) * 1000)
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)

print("\n\nFirst text chunk (ms):", first_chunk_ms)
print("Total client time (ms):", round((time.perf_counter() - started) * 1000))


## 9. Mini-reto

Crea una actualización para un **director de operaciones**: máximo 4 líneas, impacto de negocio, siguiente acción y sin inventar causa raíz. Mantén una temperatura baja.


In [ ]:
executive = run_experiment(
    "Director de operaciones",
    "Prepara una actualización ejecutiva:\n\n" + incident,
    system_prompt=(
        "Responde para un director de operaciones. Máximo 4 líneas. "
        "Explica impacto de negocio y siguiente acción. No afirmes una causa raíz si no está demostrada."
    ),
    temperature=0.1,
    max_tokens=140,
)

print(executive["text"])
print("\nTokens:", executive["input_tokens"], "+", executive["output_tokens"])
print("Latency:", executive["bedrock_latency_ms"], "ms")
print("Stop reason:", executive["stop_reason"])


# Conclusiones

Hemos observado directamente:

`system prompt + user message + contexto + parámetros → Bedrock Converse → respuesta + tokens + latencia + stopReason`

En M01-P02 pasaremos de texto libre a **salidas estructuradas** y construiremos la primera funcionalidad visible de `Analyze`.
